# Week 12. FastAPI와 Jinja2 템플릿 분리

## 제출자 정보

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

## 학습 목표

이번 주차는 FastAPI가 JSON API만 만드는 도구가 아니라, HTML 페이지도 반환할 수 있다는 점을 학습합니다.

특히 중요한 흐름은 다음 3단계입니다.

1. HTML 문자열을 Python 함수에서 직접 반환
2. 템플릿 문자열에 데이터를 주입
3. HTML, CSS, JavaScript를 각각 `templates/`, `static/` 폴더로 분리

실제 프로젝트에서는 3단계 구조가 유지보수에 가장 유리합니다.
이 노트북은 외부 패키지가 없는 환경에서도 실행되도록 표준 라이브러리 기반 예제로 원리를 설명합니다.

## 1. Step 1: HTML 문자열 직접 반환

가장 단순한 방식은 라우트 함수가 HTML 문자열을 그대로 반환하는 것입니다.
이 방식은 빠르게 테스트하기 좋지만, HTML이 길어지면 Python 코드가 지저분해집니다.

아래 함수는 FastAPI의 `@app.get("/")` 함수가 반환할 HTML을 단순화한 예입니다.

In [1]:
def render_home_direct() -> str:
    html = '''
    <html lang="ko">
      <head><title>Playlist</title></head>
      <body>
        <h1>Classical Playlist</h1>
        <p>Python 함수가 HTML 문자열을 직접 반환합니다.</p>
      </body>
    </html>
    '''
    return html.strip()

home_html = render_home_direct()
assert "<h1>Classical Playlist</h1>" in home_html
home_html

'<html lang="ko">\n      <head><title>Playlist</title></head>\n      <body>\n        <h1>Classical Playlist</h1>\n        <p>Python 함수가 HTML 문자열을 직접 반환합니다.</p>\n      </body>\n    </html>'

## 2. Step 2: 템플릿 문자열에 데이터 주입

두 번째 단계는 HTML 구조 안에 변수를 넣는 방식입니다.
Jinja2에서는 `{{ title }}` 같은 문법을 사용하지만, 여기서는 실행 안정성을 위해 표준 라이브러리 `string.Template`로 같은 개념을 설명합니다.

핵심은 HTML을 고정 문자열로 두고, 요청마다 달라지는 데이터만 주입한다는 점입니다.

In [2]:
from string import Template

now_playing_template = Template('''
<section class="now-playing">
  <p>지금 재생 중</p>
  <h2>$title</h2>
  <p>$composer</p>
</section>
''')

def render_now_playing(piece: dict) -> str:
    return now_playing_template.substitute(
        title=piece["title"],
        composer=piece["composer"],
    ).strip()

piece = {"title": "Moonlight Sonata", "composer": "Beethoven"}
now_playing_html = render_now_playing(piece)

assert "Moonlight Sonata" in now_playing_html
assert "Beethoven" in now_playing_html
now_playing_html

'<section class="now-playing">\n  <p>지금 재생 중</p>\n  <h2>Moonlight Sonata</h2>\n  <p>Beethoven</p>\n</section>'

## 3. Step 3: 템플릿 파일과 정적 파일 분리

실제 프로젝트에서는 다음처럼 역할을 나누는 것이 좋습니다.

| 구성 | 역할 |
|---|---|
| `main.py` | 라우팅, 데이터 준비, 템플릿 호출 |
| `templates/base.html` | 공통 레이아웃 |
| `templates/playlist.html` | 플레이리스트 목록 화면 |
| `templates/piece.html` | 곡 상세 화면 |
| `static/style.css` | 디자인 |
| `static/script.js` | 브라우저 상호작용 |

Python은 데이터와 라우팅에 집중하고, 화면 표현은 HTML/CSS/JS로 분리합니다.

In [3]:
from pathlib import Path
import tempfile

project_dir = Path(tempfile.mkdtemp(prefix="week12_fastapi_jinja2_"))
templates_dir = project_dir / "templates"
static_dir = project_dir / "static"
templates_dir.mkdir()
static_dir.mkdir()

files = {
    "main.py": "FastAPI app and route functions",
    "templates/base.html": "base layout with navigation",
    "templates/playlist.html": "playlist page extending base.html",
    "templates/piece.html": "piece detail page extending base.html",
    "static/style.css": "page style",
    "static/script.js": "copy interaction",
}

for relative_path, description in files.items():
    path = project_dir / relative_path
    path.write_text(description, encoding="utf-8")

created_files = sorted(str(path.relative_to(project_dir)) for path in project_dir.rglob("*") if path.is_file())

assert created_files == sorted(files)
created_files

['main.py',
 'static/script.js',
 'static/style.css',
 'templates/base.html',
 'templates/piece.html',
 'templates/playlist.html']

## 4. Jinja2의 핵심 문법을 Python 데이터로 이해하기

Jinja2 템플릿의 핵심은 변수 출력, 조건문, 반복문입니다.
아래 데이터는 `playlist.html`에 넘겨질 컨텍스트라고 볼 수 있습니다.

- `{{ playlist_name }}`: 플레이리스트 이름 출력
- `{% for piece in pieces %}`: 곡 목록 반복
- `{% if piece.id == featured_id %}`: 추천 곡 표시

In [4]:
pieces = [
    {"id": 1, "title": "Moonlight Sonata", "composer": "Beethoven", "duration": 17},
    {"id": 2, "title": "Spring", "composer": "Vivaldi", "duration": 11},
    {"id": 3, "title": "Bolero", "composer": "Ravel", "duration": 15},
]

context = {
    "playlist_name": "Classical Study Playlist",
    "pieces": pieces,
    "featured_id": 2,
}

def make_playlist_rows(context: dict) -> list[dict]:
    rows = []
    for piece in context["pieces"]:
        rows.append({
            "title": piece["title"],
            "composer": piece["composer"],
            "duration": piece["duration"],
            "featured": piece["id"] == context["featured_id"],
        })
    return rows

playlist_rows = make_playlist_rows(context)

assert sum(row["featured"] for row in playlist_rows) == 1
playlist_rows

[{'title': 'Moonlight Sonata',
  'composer': 'Beethoven',
  'duration': 17,
  'featured': False},
 {'title': 'Spring', 'composer': 'Vivaldi', 'duration': 11, 'featured': True},
 {'title': 'Bolero', 'composer': 'Ravel', 'duration': 15, 'featured': False}]

## 5. 상세 페이지 라우팅 로직

`/piece/{piece_id}` 같은 경로는 URL의 일부를 함수 인자로 받습니다.
상세 페이지에서는 해당 id의 데이터가 있으면 상세 정보를 보여주고, 없으면 안내 메시지를 표시해야 합니다.

이 분기 처리는 템플릿의 `{% if piece %}`와 연결됩니다.

In [5]:
def find_piece(piece_id: int) -> dict | None:
    return next((piece for piece in pieces if piece["id"] == piece_id), None)

def piece_detail_context(piece_id: int) -> dict:
    return {
        "piece_id": piece_id,
        "piece": find_piece(piece_id),
    }

found_context = piece_detail_context(1)
missing_context = piece_detail_context(99)

assert found_context["piece"]["title"] == "Moonlight Sonata"
assert missing_context["piece"] is None
found_context

{'piece_id': 1,
 'piece': {'id': 1,
  'title': 'Moonlight Sonata',
  'composer': 'Beethoven',
  'duration': 17}}

## 6. FastAPI 코드로 연결할 때의 구조

실제 `main.py`는 다음 역할을 합니다.

1. `app = FastAPI()`로 앱 객체 생성
2. `app.mount("/static", StaticFiles(...))`로 정적 파일 연결
3. `Jinja2Templates(directory="templates")`로 템플릿 폴더 지정
4. 라우트 함수에서 `TemplateResponse` 반환

이 구조를 지키면 HTML 화면이 커져도 Python 라우팅 코드를 크게 바꾸지 않아도 됩니다.

In [6]:
fastapi_structure = {
    "app": "FastAPI instance",
    "static": "CSS and JavaScript files served from /static",
    "templates": "HTML files rendered with context data",
    "routes": ["/", "/now-playing", "/playlist", "/piece/{piece_id}"],
}

assert "/playlist" in fastapi_structure["routes"]
assert fastapi_structure["templates"].startswith("HTML")
fastapi_structure

{'app': 'FastAPI instance',
 'static': 'CSS and JavaScript files served from /static',
 'templates': 'HTML files rendered with context data',
 'routes': ['/', '/now-playing', '/playlist', '/piece/{piece_id}']}

## 정리

12주차의 핵심은 **관심사 분리**입니다.

- Python: 라우팅과 데이터 준비
- HTML: 문서 구조
- CSS: 디자인
- JavaScript: 브라우저 동작

FastAPI와 Jinja2를 함께 쓰면 JSON API와 HTML 페이지를 같은 백엔드에서 다룰 수 있습니다.